# LLM as Judge: Using an LLM to Evaluate Another LLM's Output

This example demonstrates the "LLM as a Judge" pattern:
1. An LLM generates a response to a question
2. A second LLM call evaluates/judges that response
3. The judge scores the response on specific criteria

This is a fundamental quality control mechanism for AI pipelines.
Before you ship an LLM's output to a user, you can have another LLM
check it first.

**Pipeline:**
```
[Question] --> [Generator LLM] --> [Response] --> [Judge LLM] --> [Scores + Feedback]
```

For production evaluations, we STRONGLY RECOMMEND using the DeepEval
framework instead of rolling your own judge. See `deepeval_example.ipynb`.

## Install Dependencies

In [ ]:
!pip install anthropic

## Imports and Configuration

Use the same model for both generator and judge in this demo.
In production, consider using a STRONGER model as the judge.
For example: Haiku generates, Sonnet judges. Or Sonnet generates, Opus judges.

In [ ]:
import json
import os

import anthropic


client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# Use the same model for both generator and judge in this demo.
# In production, consider using a STRONGER model as the judge.
# For example: Haiku generates, Sonnet judges. Or Sonnet generates, Opus judges.
GENERATOR_MODEL = "claude-sonnet-4-20250514"
JUDGE_MODEL = "claude-sonnet-4-20250514"

## Stage 1: Generator -- Produce a Response

This is the "generator" -- the LLM whose output we want to evaluate.
In a real pipeline, this could be a RAG system, a chatbot, a content
generator, or any LLM-powered component.

In [ ]:
def generate_response(question: str, context: str = "") -> str:
    """
    Generate a response to a question using an LLM.

    This is the "generator" — the LLM whose output we want to evaluate.
    In a real pipeline, this could be a RAG system, a chatbot, a content
    generator, or any LLM-powered component.

    Args:
        question: The question to answer
        context: Optional context/facts the LLM should use

    Returns:
        The generated response as a string
    """

    print("=" * 60)
    print("STAGE 1: Generator")
    print(f"Question: {question}")
    print("=" * 60)

    messages = []

    if context:
        messages.append({
            "role": "user",
            "content": f"""Answer the following question using ONLY the provided context.
If the context doesn't contain enough information, say so.

CONTEXT:
{context}

QUESTION: {question}

Provide a helpful, accurate answer."""
        })
    else:
        messages.append({
            "role": "user",
            "content": f"Answer this question helpfully and accurately: {question}"
        })

    response = client.messages.create(
        model=GENERATOR_MODEL,
        max_tokens=1024,
        messages=messages
    )

    generated_text = response.content[0].text
    print(f"\nGenerated response ({len(generated_text)} chars):")
    print(f"  {generated_text[:200]}...")
    print()

    return generated_text

## Stage 2: Judge -- Evaluate the Response

This is the core of the "LLM as a Judge" pattern. The judge LLM receives:
1. The original question
2. The generated response to evaluate
3. Any context/facts that should have been used
4. Explicit scoring criteria (the rubric)

The judge prompt is structured to be specific and measurable.
Vague criteria like "is it good?" produce unreliable judgments.

In [ ]:
def judge_response(
    question: str,
    response: str,
    context: str = "",
    criteria: dict = None
) -> dict:
    """
    Use an LLM to judge/evaluate a generated response.

    This is the core of the "LLM as a Judge" pattern. The judge LLM
    receives:
        1. The original question
        2. The generated response to evaluate
        3. Any context/facts that should have been used
        4. Explicit scoring criteria (the rubric)

    Args:
        question: The original question that was asked
        response: The generated response to evaluate
        context: The context/facts the response should be based on
        criteria: Custom scoring criteria (uses defaults if not provided)

    Returns:
        A dict with scores and feedback
    """

    print("=" * 60)
    print("STAGE 2: Judge")
    print("=" * 60)

    # Default scoring criteria if none provided
    if criteria is None:
        criteria = {
            "accuracy": "Are the facts correct? Does the response contain any incorrect information? (1-5)",
            "relevance": "Does the response actually answer the question that was asked? (1-5)",
            "completeness": "Does the response cover all important aspects of the question? (1-5)",
            "clarity": "Is the response well-organized, clear, and easy to understand? (1-5)",
            "faithfulness": "Does the response stick to the provided context without adding unsupported claims? (1-5)"
        }

    criteria_text = "\n".join(
        f"- {name}: {description}"
        for name, description in criteria.items()
    )

    # Build the judge prompt
    # KEY: The judge prompt is structured to be specific and measurable.
    # Vague criteria like "is it good?" produce unreliable judgments.
    judge_prompt = f"""You are a strict but fair evaluator. Your job is to score an AI-generated
response on specific criteria. Be honest and critical — don't give high scores
unless they are truly deserved.

SCORING CRITERIA:
{criteria_text}

ORIGINAL QUESTION:
{question}

"""

    if context:
        judge_prompt += f"""CONTEXT (facts the response should be based on):
{context}

"""

    judge_prompt += f"""RESPONSE TO EVALUATE:
{response}

INSTRUCTIONS:
1. Carefully read the question, context (if provided), and response.
2. Score each criterion from 1 (terrible) to 5 (excellent).
3. Provide a brief justification for each score.
4. Provide an overall assessment.

Return ONLY valid JSON in this exact format (no markdown, no code fences):
{{
    "scores": {{
        "accuracy": {{"score": 1-5, "justification": "why this score"}},
        "relevance": {{"score": 1-5, "justification": "why this score"}},
        "completeness": {{"score": 1-5, "justification": "why this score"}},
        "clarity": {{"score": 1-5, "justification": "why this score"}},
        "faithfulness": {{"score": 1-5, "justification": "why this score"}}
    }},
    "overall_score": 1-5,
    "overall_assessment": "2-3 sentence summary of the evaluation",
    "pass": true/false,
    "issues_found": ["list any specific problems found, or empty list if none"]
}}

Set "pass" to true if overall_score >= 4, false otherwise."""

    response_obj = client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=1024,
        messages=[
            {"role": "user", "content": judge_prompt}
        ]
    )

    raw_output = response_obj.content[0].text

    try:
        judgment = json.loads(raw_output)
    except json.JSONDecodeError:
        judgment = {
            "scores": {},
            "overall_score": 0,
            "overall_assessment": f"Failed to parse judge output: {raw_output[:200]}",
            "pass": False,
            "issues_found": ["Judge output was not valid JSON"]
        }

    # Print the judgment
    print(f"\nOverall Score: {judgment.get('overall_score', 'N/A')}/5")
    print(f"Pass: {'YES' if judgment.get('pass') else 'NO'}")
    print(f"\nScores by criteria:")
    for criterion, details in judgment.get("scores", {}).items():
        if isinstance(details, dict):
            print(f"  {criterion}: {details.get('score', 'N/A')}/5 — {details.get('justification', 'N/A')}")
    print(f"\nOverall assessment: {judgment.get('overall_assessment', 'N/A')}")

    if judgment.get("issues_found"):
        print(f"\nIssues found:")
        for issue in judgment["issues_found"]:
            print(f"  - {issue}")
    print()

    return judgment

## Full LLM-as-Judge Pipeline

Run the full generate-then-judge pipeline.

**Flow:**
```
Question --> Generator --> Response --> Judge --> Scores
```

In [ ]:
def run_judge_pipeline(
    question: str,
    context: str = "",
    criteria: dict = None
) -> dict:
    """
    Run the full generate-then-judge pipeline.

    Flow:
        Question --> Generator --> Response --> Judge --> Scores

    Returns both the generated response and the judgment.
    """

    print("\n" + "#" * 60)
    print("# LLM-AS-JUDGE PIPELINE START")
    print("#" * 60 + "\n")

    # Stage 1: Generate a response
    generated_response = generate_response(question, context)

    # Stage 2: Judge the response
    judgment = judge_response(question, generated_response, context, criteria)

    # Combine results
    result = {
        "question": question,
        "context_provided": bool(context),
        "generated_response": generated_response,
        "judgment": judgment
    }

    print("#" * 60)
    verdict = "PASSED" if judgment.get("pass") else "FAILED"
    print(f"# PIPELINE COMPLETE — Verdict: {verdict}")
    print("#" * 60)

    return result

## Iterative Refinement Pattern (Bonus)

Generate a response and refine it until it passes the judge.

This is the "loop pipeline" pattern:
```
Generate --> Judge --> [if fail] --> Regenerate with feedback --> Judge --> ...
```

This is powerful because each retry includes the judge's feedback,
so the generator can fix specific issues.

In [ ]:
def generate_with_refinement(
    question: str,
    context: str = "",
    max_attempts: int = 3,
    passing_score: int = 4
) -> dict:
    """
    Generate a response and refine it until it passes the judge.

    This is the "loop pipeline" pattern:
        Generate --> Judge --> [if fail] --> Regenerate with feedback --> Judge --> ...

    This is powerful because each retry includes the judge's feedback,
    so the generator can fix specific issues.

    Args:
        question: The question to answer
        context: Context/facts to use
        max_attempts: Maximum generation attempts
        passing_score: Minimum overall score to pass (1-5)
    """

    print("\n" + "#" * 60)
    print("# ITERATIVE REFINEMENT PIPELINE")
    print(f"# Max attempts: {max_attempts}, Passing score: {passing_score}/5")
    print("#" * 60 + "\n")

    previous_feedback = ""

    for attempt in range(1, max_attempts + 1):
        print(f"\n--- Attempt {attempt}/{max_attempts} ---\n")

        # Generate (with feedback from previous attempt if available)
        if previous_feedback:
            enhanced_question = f"""{question}

IMPORTANT: A previous version of your response was evaluated and received
this feedback. Please address these issues:
{previous_feedback}"""
        else:
            enhanced_question = question

        generated_response = generate_response(enhanced_question, context)

        # Judge
        judgment = judge_response(question, generated_response, context)

        overall_score = judgment.get("overall_score", 0)

        if overall_score >= passing_score:
            print(f"\nResponse PASSED on attempt {attempt} with score {overall_score}/5")
            return {
                "question": question,
                "final_response": generated_response,
                "judgment": judgment,
                "attempts": attempt,
                "passed": True
            }
        else:
            # Extract feedback for the next attempt
            issues = judgment.get("issues_found", [])
            assessment = judgment.get("overall_assessment", "")
            previous_feedback = f"Assessment: {assessment}\nIssues: {'; '.join(issues)}"
            print(f"\nResponse FAILED (score: {overall_score}/5). Retrying with feedback...")

    print(f"\nFailed to produce passing response after {max_attempts} attempts.")
    return {
        "question": question,
        "final_response": generated_response,
        "judgment": judgment,
        "attempts": max_attempts,
        "passed": False
    }

## Example 1: Judging a Context-Based Response (like RAG)

In [ ]:
result1 = run_judge_pipeline(
    question="What is the average price per square foot for homes in Kailua, Hawaii?",
    context="""According to Hawaii MLS data from Q4 2025:
- Kailua median home price: $1,250,000
- Average price per square foot in Kailua: $785
- Average home size in Kailua: 1,592 sq ft
- Year-over-year price change: +3.8%
- Average days on market: 28 days
- Total active listings: 45"""
)

## Example 2: Judging a General Knowledge Response

In [ ]:
result2 = run_judge_pipeline(
    question="What are the key factors a first-time home buyer in Hawaii should consider?",
)

## Example 3: Iterative Refinement Until Passing

In [ ]:
result3 = generate_with_refinement(
    question="Write a brief market update email for Kailua homeowners thinking about selling.",
    context="""Q4 2025 Kailua Market Data:
- Median sale price: $1,250,000 (up 3.8% YoY)
- Average days on market: 28 (down from 35 last year)
- Months of inventory: 1.8 (strong seller's market)
- Number of sales this quarter: 87
- Price per sq ft: $785""",
    max_attempts=2,
    passing_score=4
)

## Production Recommendation: Use DeepEval

The LLM-as-Judge pattern shown above is great for understanding
the concept, but for production systems, we strongly recommend
using the DeepEval framework instead of rolling your own judge.

DeepEval provides:
- Pre-built, tested metrics (hallucination, relevancy, etc.)
- Pytest integration for CI/CD
- Dashboard for tracking eval results over time
- Battle-tested by real companies

See `deepeval_example.ipynb` for how to use it.

```
pip install deepeval
```